In [2]:
import pandas as pd

In [4]:
df = pd.read_json(r"../data/laptops_data.json")
df.head(2)

,url,name,price,images,specifications
0,https://www.paklap.pk/infinix-inbook-air-xl442...,Infinix InBook Air XL442 - Alder Lake - 12th G...,"Rs. 99,900.00",[https://www.paklap.pk/media/catalog/category/...,"{'Brand': 'Infinix', 'Generation': '12th Gener..."
1,https://www.paklap.pk/hp-15-fd0532nia-13th-gen...,HP 15 FD0532nia - Raptor Lake - 13th Gen Core ...,"Rs. 105,000.00",[https://www.paklap.pk/media/catalog/category/...,"{'Brand': 'Hp', 'Generation': '13th Generation..."


In [5]:
len(df)

91

# Remove Duplicate Images

Remove images that appear in multiple laptop entries. If an image appears in more than one laptop, it will be removed from all entries.

In [9]:
# Count image occurrences across all laptops
from collections import Counter

# Collect all images
all_images = []
for images_list in df['images']:
    all_images.extend(images_list)

# Count occurrences
image_counts = Counter(all_images)

# Find duplicate images (appearing in more than one laptop entry)
duplicate_images = {img for img, count in image_counts.items() if count > 1}

print(f"Total images: {len(all_images)}")
print(f"Unique images: {len(set(all_images))}")
print(f"Duplicate images (appearing in multiple laptops): {len(duplicate_images)}")
print(f"\nDuplicate images to remove:")
for img in sorted(duplicate_images):
    print(f"  - {img} (appears {image_counts[img]} times)")

Total images: 237
Unique images: 57
Duplicate images (appearing in multiple laptops): 2

Duplicate images to remove:
  - https://www.paklap.pk/media/catalog/category/laptop-price-in-pakistan.jpg (appears 91 times)
  - https://www.paklap.pk/media/catalog/category/used-laptop-price-in-pakistan.jpg (appears 91 times)


In [10]:
# Remove duplicate images from all laptop entries
def remove_duplicate_images(images_list, duplicate_set):
    """Remove images that appear in the duplicate set"""
    return [img for img in images_list if img not in duplicate_set]

# Apply the filter to all laptops
df['images'] = df['images'].apply(lambda x: remove_duplicate_images(x, duplicate_images))

# Check results
total_images_after = sum(len(images) for images in df['images'])
print(f"Images before: {len(all_images)}")
print(f"Images after: {total_images_after}")
print(f"Images removed: {len(all_images) - total_images_after}")

# Show sample of cleaned data
print("\nSample of cleaned image lists:")
for idx, row in df.head(3).iterrows():
    print(f"\n{row['name'][:50]}...")
    print(f"  Images: {len(row['images'])} remaining")
    for img in row['images']:
        print(f"    - {img}")

Images before: 237
Images after: 55
Images removed: 182

Sample of cleaned image lists:

Infinix InBook Air XL442 - Alder Lake - 12th Gen C...
  Images: 1 remaining
    - https://www.paklap.pk/media/catalog/product/cache/2cc443e44e97595ea39006016c876eaa/i/n/infinix_inbook_air_xl442_core_i3_laptop_pakistan_2_.jpg

HP 15 FD0532nia - Raptor Lake - 13th Gen Core i3 1...
  Images: 1 remaining
    - https://www.paklap.pk/media/catalog/product/cache/2cc443e44e97595ea39006016c876eaa/h/p/hp_laptop_15_fd0532nia_laptop_pakistan_1_.jpg

Lenovo V15 G4 - Raptor Lake - 13th Gen Core i3 Hex...
  Images: 1 remaining
    - https://www.paklap.pk/media/catalog/product/cache/2cc443e44e97595ea39006016c876eaa/l/e/lenovo_v15_g4_laptop_pakistan_7__1.jpg


In [11]:
# Save the cleaned data back to JSON
import json

# Convert DataFrame back to list of dictionaries
cleaned_data = df.to_dict('records')

# Save to file
output_path = r"../data/laptops__updated_data.json"
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(cleaned_data, f, indent=2, ensure_ascii=False)

print(f"Cleaned data saved to {output_path}")
print(f"Total laptops: {len(cleaned_data)}")

Cleaned data saved to ../data/laptops__updated_data.json
Total laptops: 91


In [ ]:
# Analyze laptops with no images
df_updated = pd.read_json(r"../data/laptops__all_images.json")

# Count laptops with empty image lists
laptops_no_images = df_updated[df_updated['images'].apply(lambda x: len(x) == 0)]
num_no_images = len(laptops_no_images)

print(f"Total laptops in updated file: {len(df_updated)}")
print(f"Laptops with no images: {num_no_images}")
print(f"Percentage: {(num_no_images / len(df_updated) * 100):.2f}%")

if num_no_images > 0:
    print(f"\nLaptops with no images:")
    for idx, row in laptops_no_images.iterrows():
        print(f"  - {row['name'][:70]}...")
        print(f"    URL: {row['url']}")
        print(f"    Price: {row['price']}")
        print()

Total laptops in updated file: 91
Laptops with no images: 0
Percentage: 0.00%


In [16]:
# Convert price from string to float
import re

def convert_price_to_float(price_str):
    """
    Convert price string like "Rs. 99,900.00" to float 99900.00
    """
    if pd.isna(price_str) or price_str == "":
        return None
    
    # Remove "Rs." and any whitespace
    price_clean = price_str.replace("Rs.", "").strip()
    
    # Remove commas
    price_clean = price_clean.replace(",", "")
    
    # Convert to float
    try:
        return float(price_clean)
    except ValueError:
        print(f"Warning: Could not convert price: {price_str}")
        return None

# Apply conversion
df_updated['price'] = df_updated['price'].apply(convert_price_to_float)

# Show sample results
print("Price conversion results:")
print(f"Data type: {df_updated['price'].dtype}")
print(f"\nSample prices:")
print(df_updated[['name', 'price']].head(10))

# Check for any failed conversions
failed_conversions = df_updated[df_updated['price'].isna()]
if len(failed_conversions) > 0:
    print(f"\n{len(failed_conversions)} laptops with failed price conversion")
else:
    print(f"\nAll {len(df_updated)} prices converted successfully!")

# Save to final JSON file
final_data = df_updated.to_dict('records')
final_output_path = r"../data/Final_laptops_data.json"

with open(final_output_path, 'w', encoding='utf-8') as f:
    json.dump(final_data, f, indent=2, ensure_ascii=False)

print(f"\nFinal data saved to {final_output_path}")
print(f"Total laptops: {len(final_data)}")

Price conversion results:
Data type: float64

Sample prices:
                                                name     price
0  Infinix InBook Air XL442 - Alder Lake - 12th G...   99900.0
1  HP 15 FD0532nia - Raptor Lake - 13th Gen Core ...  105000.0
2  Lenovo V15 G4 - Raptor Lake - 13th Gen Core i3...  105900.0
3  Lenovo IdeaPad Slim 3 15 - Raptor Lake - 13th ...  107500.0
4  Dell Vostro 15 3530 - Raptor Lake  - 13th Gen ...  108999.0
5  HP EliteBook 640 G10 Notebook PC - Raptor Lake...  110000.0
6  Lenovo IdeaPad 3 15 - Tiger Lake - 11th Gen Co...  122500.0
7  Lenovo V15 G5 - Raptor Lake - 13th Gen Core i3...  123500.0
8  Lenovo IdeaPad Slim 3 15 - Raptor Lake - 13th ...  132000.0
9  Asus VivoBook 15 X1504VA - Raptor Lake - Intel...  137500.0

All 91 prices converted successfully!

Final data saved to ../data/Final_laptops_data.json
Total laptops: 91
